In [1]:
import pandas as pd
import numpy as np
from pandas import Series, DataFrame

import requests
from io import StringIO

In [2]:
r = requests.get('https://api.blockchain.info/charts/market-price?format=csv')


In [3]:
s = StringIO(r.content.decode())

In [4]:
df = pd.read_csv(
    s,
    header= None,
    names= ['date', 'price']
)

In [5]:
df

,date,price
0,2025-03-12 00:00:00,82859.32
1,2025-03-13 00:00:00,83703.69
2,2025-03-14 00:00:00,81075.63
3,2025-03-15 00:00:00,83982.58
4,2025-03-16 00:00:00,84339.61
...,...,...
361,2026-03-08 00:00:00,67279.51
362,2026-03-09 00:00:00,65940.80
363,2026-03-10 00:00:00,68452.78
364,2026-03-11 00:00:00,69922.38


In [6]:
#for digit grouping and control floating-point numbers

pd.set_option('display.float_format', '{:,.2f}'.format)

# The closing price for the most recent trading day

df.tail(1)[['price']]

,price
365,"70,197.35"


In [7]:
# The lowest historical price and the date of that price

df.loc[df['price'] == df['price'].min(), ['date', 'price']]

,date,price
331,2026-02-06 00:00:00,"62,812.06"


In [8]:
# The highest historical price and the date of that price

df.loc[df['price'] == df['price'].max(), ['date', 'price']]

,date,price
209,2025-10-07 00:00:00,"124,776.68"


In [9]:
# Reuven elegant solution for the date of highest and lowest price

df.set_index('date').idxmin(), df.set_index('date').idxmax()

(price    2026-02-06 00:00:00
 dtype: str,
 price    2025-10-07 00:00:00
 dtype: str)

In [10]:
# Reuven more elegant solution

df.set_index('date').agg(['idxmin', 'idxmax'])

,price
idxmin,2026-02-06 00:00:00
idxmax,2025-10-07 00:00:00


### Beyond the exercise_1

##### In this exercise, you downloaded the information into a data frame and then performed calculations on it. Without assigning the downloaded data to an interim variable, can you return the current value? Your solution should consist of a single line of code that includes the download, selection, and calculation.

In [11]:
df = pd.read_csv(
    'https://api.blockchain.info/charts/market-price?format=csv',
    header= None,
    names= ['date', 'price']
).set_index('date').agg(['idxmin', 'idxmax'])

In [12]:
df

,price
idxmin,2026-02-06 00:00:00
idxmax,2025-10-07 00:00:00


In [13]:
pd.read_csv(
    'https://api.blockchain.info/charts/market-price?format=csv',
    header= None,
    names= ['date', 'price']
).set_index('date').agg(['idxmin', 'idxmax'])

,price
idxmin,2026-02-06 00:00:00
idxmax,2025-10-07 00:00:00


### Beyond the exercise_2


##### The `pd.read_html` function, `like pd.read_csv`, takes a file-like object or a URL. It assumes that it will encounter HTML-formatted text containing at least one table. It turns each table into a data frame and then returns a list of those data frames. With this in mind, retrieve one year of historical S&P 500 data from Yahoo Finance (https://finance.yahoo.com/quote/%5EGSPC/history?p=%5EGSPC), looking only at the `Date`, `Close`, and `Volume` columns. Show the date and volume of the days with the highest and lowest `Close` values. Note that Yahoo seems to look at the `User-Agent` header in the HTTP request, which cannot be set in `read_html`. So you’ll need to use requests to retrieve the data, setting User-Agent to a string equal to `'Mozilla 5.0'`. Turn the content of the result into a `StringIO`, and then feed that to `read_html` and retrieve the data.

In [14]:
# Reuven's solution didn't work so I used the yfinance package

import yfinance as yf

df = yf.download("^GSPC", start="2023-01-01", end="2026-03-11")

df.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
2023-01-03,"3,824.14","3,878.46","3,794.33","3,853.29",3959140000
2023-01-04,"3,852.97","3,873.16","3,815.77","3,840.36",4414080000
2023-01-05,"3,808.10","3,839.74","3,802.42","3,839.74",3893450000
2023-01-06,"3,895.08","3,906.19","3,809.56","3,823.37",3923560000
2023-01-09,"3,892.09","3,950.57","3,890.42","3,910.82",4311770000


In [15]:
df.columns = df.columns.droplevel(1) # There's a MultiIndex structure in the columns and I want to drop the ^GSPC designation

df

Price,Close,High,Low,Open,Volume
Date,,,,,
2023-01-03,"3,824.14","3,878.46","3,794.33","3,853.29",3959140000
2023-01-04,"3,852.97","3,873.16","3,815.77","3,840.36",4414080000
2023-01-05,"3,808.10","3,839.74","3,802.42","3,839.74",3893450000
2023-01-06,"3,895.08","3,906.19","3,809.56","3,823.37",3923560000
2023-01-09,"3,892.09","3,950.57","3,890.42","3,910.82",4311770000
...,...,...,...,...,...
2026-03-04,"6,869.50","6,885.94","6,811.64","6,831.69",5252170000
2026-03-05,"6,830.71","6,870.43","6,770.78","6,851.08",5989300000
2026-03-06,"6,740.02","6,773.42","6,711.56","6,769.03",5793120000


### Beyond the exercise_3


#### Create a two-row data frame with the highest and lowest closing prices for the S&P 500. Use the `to_csv` function to write this data to a new CSV file.

In [16]:
list(df['Close'].agg(['min', 'max']))

[3808.10009765625, 6978.60009765625]

In [17]:
filtered = df.loc[df['Close'].isin(df['Close'].agg(['min', 'max']))][['Close']]
filtered

Price,Close
Date,
2023-01-05,"3,808.10"
2026-01-27,"6,978.60"


In [18]:
filtered.to_csv('ch-3_beyond-ex-3.csv')